# TRPG 룰북 RAG — 정리본 (검색 v1 고정 + 설명 품질 개선)

**새 Colab 노트북에서 위에서 아래로 순서대로 실행**하면 돼요. 런타임 재시작 후에도 똑같이 위에서부터 돌리면 됩니다.

| 구간 | 내용 | 얼마나 걸려요 |
|---|---|---|
| 0~2 | 설치, Ollama, 모델 받기, 경로 | 처음 10~15분 (모델 다운로드), 이후엔 빨라요 |
| 3~8 | 청크·임베딩·FAISS·평가도구·번역·BM25·검색 (**검색 v1, 건드리지 않음**) | 임베딩 캐시가 있으면 1분 |
| 9~10 | 답변 생성 도구(14B), `rag_answer`, 질문해 보기 | 바로 |
| 8 | 번역·검색 캐시 (개발셋은 번역·임베딩 호출 없이) — **개발셋 점수 확인 포함** | 처음 한 번만 몇 분, 이후 불러오기만 |
| 11~12 | 개발셋 검색 평가, 설명 품질 실험 도구 (개발셋 38문항) | 바로 |
| 13 | 근거 선택 실험 (1단계 방식 vs 2단계 방식) | 캐시가 있으면 몇 분 |
| 15~16 | **2단계 답변 파이프라인** (선택 → 검증/거절 → 설명 → 자동 검사 → 근거 충실성) + `dev_view` + 걸림 비율 측정 | 개발셋 38문항 몇 분 |
| 14(끝) | 최종 세트(golden_final) — **잠가 둔 셀**, 함부로 돌리지 않아요 | |

**검색 v1 정의 (바꾸면 안 되는 것)**
- 번역①: 힌트 없는 일반 번역(예시 3개), 번역②: 규칙 용어 27개 선택지 번역
- 번역·설명·근거 선택 모두 **qwen2.5:14b 하나**로 해요. 번역 호출은 `temperature 0, seed 42, num_ctx 4096`
- ⚠️ 옛 v1은 **7B 번역**으로 쟀어요. 14B 번역으로 바꾸면 검색 결과가 달라질 수 있어서, 8번 구간에서 개발셋 점수를 다시 확인해요 (7B 기준 개발 66문항: 41/54/57)
- 임베딩 3갈래(원문 질문, 번역①, 번역②) 비중 1 + BM25 2갈래(번역①, ②) 비중 0.5 → 가중 RRF → 5개
- 최종 세트 옛 v1(7B 번역) 점수: 1등 31/60, 3등 안 46/60, 5등 안 51/60 (14B 번역 버전은 아직 안 쟀어요 → v2로 따로)

**모델은 qwen2.5:14b 하나**예요 (임베딩은 bge-m3).

## 0. 설치

In [ ]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
faiss-cpu==1.15.1

In [ ]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|faiss"

## 1. Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

In [ ]:
import os, re, json, time, math, subprocess
from collections import Counter
import numpy as np, requests, faiss
from tqdm.auto import tqdm
from google.colab import drive, files

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"
GEN_MODEL = "qwen2.5:14b"            # 모델은 이것 하나만 써요 (번역·설명·근거 선택 전부)
TRANSLATE_MODEL = GEN_MODEL

def wait_server(url, seconds=120):
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")

os.environ["OLLAMA_KEEP_ALIVE"] = "1h"       # 모델을 1시간 메모리에 유지 (서버를 켜기 전에 설정해야 해요)
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
wait_server(OLLAMA_URL)

In [ ]:
# 모델 2개(임베딩 + 14B)를 백그라운드로 차례로 받기 (이 셀은 바로 끝나요)
subprocess.Popen('nohup sh -c "ollama pull bge-m3 && ollama pull qwen2.5:14b" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

In [ ]:
# 모델이 다 받아졌는지 확인 (2개가 모두 보이면 OK. 아니면 잠시 뒤 다시 실행)
out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(out)
need = ["bge-m3", "qwen2.5:14b"]
print("준비 안 된 모델:", [m for m in need if m not in out] or "없음")

## 2. 구글 드라이브 · 경로

In [ ]:
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/KT 기본프로젝트"
OUT_PATH = f"{DATA}/rules_new.jsonl"          # 01 노트북이 만든 청크
EMB_PATH = f"{DATA}/rules_new_emb.npy"        # 임베딩 캐시
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)

## 3. 청크 읽기 · 임베딩(캐시) · FAISS 인덱스

In [ ]:
# Ex01. 저장한 청크 읽기 (PDF 셀을 안 돌려도 여기서 시작)
chunks = [json.loads(line) for line in open(OUT_PATH, encoding="utf-8")]
chunks = [c for c in chunks if c["meta"]["type"] == "rule"]     # 법적 고지(meta)는 검색 대상 아님
print("검색 대상 청크:", len(chunks))                            # 1957

In [ ]:
# Ex02. 여러 문장을 한 번에 임베딩 → .npy 캐시 (청크 수가 같으면 캐시를 그대로 씀)
def embed_batch(texts, model=EMBED_MODEL):
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["embeddings"]

BATCH = 32
if os.path.exists(EMB_PATH) and np.load(EMB_PATH).shape[0] == len(chunks):
    emb = np.load(EMB_PATH)
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(chunks), BATCH)):
        vecs += embed_batch([c["text"] for c in chunks[i:i + BATCH]])
    emb = np.array(vecs, dtype="float32")
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)                               # (1957, 1024)

In [ ]:
# Ex03. FAISS 인덱스 만들기
vecs = emb.copy()
faiss.normalize_L2(vecs)                       # 길이를 1로 → 내적 = 코사인 유사도
index = faiss.IndexFlatIP(vecs.shape[1])       # 1024차원 벡터를 내적 기준으로 정확히 비교
index.add(vecs)                                # i번째 벡터 = chunks[i]
print("인덱스에 들어간 벡터 수:", index.ntotal)               # 1957

## 4. 평가 도구와 질문 세트
정답은 질문과 한 쌍(`q`, `want`)으로 파일에 두고, 검색 함수에는 `q`만 넣고 `want`는 채점에서만 써요.

- 개발셋: `golden`(영어 25) + `golden_new`(한국어 13) = **38문항** ← 설명 개선은 여기서만
- `golden_ko`(28)는 이미 오염된 옛 개발셋 (검색 점수 참고용)
- `golden_final`(60)은 최종 측정용 → 14번 구간에서만 사용

In [ ]:
# Ex04. 정답 비교 도구 + 질문 세트 읽기
ALIAS = {"Grappl": {"grappled", "grappling"}}     # 줄여 적은 정답 보정

def norm(title):
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)      # 이름 끝의 [..] 제거
    title = re.sub(r"\s*\([^)]*\)$", "", title)       # 이름 끝의 (..) 제거
    return title.strip().lower()

def is_hit(title, want):                              # 이름이 정확히 같을 때만 정답
    return norm(title) == norm(want) or norm(title) in ALIAS.get(want, ())

def wants_of(g):
    return [g["want"]] if isinstance(g["want"], str) else g["want"]

def rank_in(found, g):                                # 검색된 청크 목록에서 정답의 순위(없으면 None)
    titles = [c["meta"]["title"] for c in found]
    rank = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
    return rank, titles

def counts(ranks):                                    # (1등, 3등 안, 5등 안) 개수
    return (sum(r == 1 for r in ranks),
            sum(r is not None and r <= 3 for r in ranks),
            sum(r is not None for r in ranks))

def evaluate(fn, items, show_miss=False):             # 검색 함수 fn(q) -> 청크 목록 의 (1등, 3등 안, 5등 안)
    ranks = []
    for g in tqdm(items, leave=False):
        found = fn(g["q"])                            # 검색에는 질문(q)만 넣음
        r, titles = rank_in(found, g)                 # 정답(want)은 채점에서만 사용
        ranks.append(r)
        if show_miss and r != 1:
            print(f"[{r}] {g['q']}\n     정답: {g['want']} | 검색: {titles[:3]}")
    return counts(ranks)

def load_set(name):
    path = f"{DATA}/{name}.json"
    return json.load(open(path, encoding="utf-8")) if os.path.exists(path) else []

golden       = load_set("golden")          # 영어 25문항
golden_ko    = load_set("golden_ko")       # 한국어 28문항 (이미 오염됨)
golden_new   = load_set("golden_new")      # 한국어 13문항
golden_final = load_set("golden_final")    # 최종 60문항 (14번 구간 전에는 점수 재지 않아요)

# 정답 제목이 룰북에 실제로 있는지 확인 (오타 방지)
titles_all = {norm(c["meta"]["title"]) for c in chunks}
for name, items in {"golden": golden, "golden_ko": golden_ko, "golden_new": golden_new, "golden_final": golden_final}.items():
    miss = [w for g in items for w in wants_of(g) if norm(w) not in titles_all and w not in ALIAS]
    print(f"{name}: {len(items)}문항 | 룰북에 없는 정답: {miss}")

In [ ]:
# Ex04-b. 선택 평가용 느슨한 제목 비교 (단수/복수만 같은 것으로 봐요). 검색 점수에는 쓰지 않아요!
def stem(title):
    return re.sub(r"s\b", "", norm(title))

def is_hit_loose(title, want):                        # 2단계 선택 평가에서만 사용 (검색 점수는 엄격한 is_hit)
    return stem(title) == stem(want) or norm(title) in ALIAS.get(want, ())

## 5. 번역 (한국어 질문 → 영어 검색어)
- 번역①: 힌트 없는 일반 번역(예시 3개)
- 번역②: 규칙 용어 27개 선택지 번역
- 호출은 **14B, temperature 0, seed 42, num_ctx 4096**. (옛 v1은 7B 번역이었고 num_ctx를 바꾸면 번역도 달라져요.)

In [ ]:
# Ex05-a. 번역 호출 (같은 입력이면 같은 출력: temperature 0 + seed 고정)
def ask_plain(prompt, model=None, temperature=0.0):
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,
              "messages": [{"role": "user", "content": prompt}],      # 한국어 고정 system 없이 호출
              "stream": False,
              "options": {"temperature": temperature, "seed": 42, "num_ctx": 4096}},   # 8192면 번역이 달라져요
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [ ]:
# Ex05-b. 번역 프롬프트 두 가지
def clean_query(s):                                   # 첫 줄만, "Query:" 접두어와 따옴표 제거
    s = s.strip().split("\n")[0]
    s = re.sub(r"^\s*query\s*:\s*", "", s, flags=re.I)
    return s.strip().strip("\"'")

# 번역① 힌트 없는 일반 번역 (예시 3개)
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 레벨업하려면 경험치가 얼마나 필요해?
Query: Experience Points needed to advance level

Question: 방어도는 어떻게 계산해?
Query: Armor Class calculation

Question: {question}
Query:"""

def rewrite_query(q):
    return clean_query(ask_plain(REWRITE_TEMPLATE.format(question=q)))

# 번역② 용어 선택지 (규칙 용어 27개 중 해당되면 그대로 쓰게 함)
RULE_TERMS = ["Blinded", "Charmed", "Deafened", "Exhaustion", "Frightened", "Grappled", "Incapacitated",
              "Invisible", "Paralyzed", "Petrified", "Poisoned", "Prone", "Restrained", "Stunned", "Unconscious",
              "Attack action", "Dash action", "Disengage action", "Dodge action", "Help action", "Hide action",
              "Influence action", "Magic action", "Ready action", "Search action", "Study action", "Utilize action"]

REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
If the question is about one of these official rule terms, use that term exactly as written.
Official terms: {terms}
Output only the query, nothing else.

Question: {q}
Query:"""

def rewrite_query2c(q):
    return clean_query(ask_plain(REWRITE2.format(terms=", ".join(RULE_TERMS), q=q)))

print(rewrite_query("겁에 질리면 뭘 못 해?"), "|", rewrite_query2c("겁에 질리면 뭘 못 해?"))

## 6. 키워드 검색 BM25 (직접 구현, 새 설치 없음)
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교해서 서로 보완돼요. 이름이 정확한 질문(`Two-Weapon Fighting`)에 강해요. 제목을 두 번 넣어 제목 단어에 가중치를 줘요.

In [ ]:
# Ex06. BM25 직접 만들기
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())

def tokenize(s):                                       # 영어 소문자 단어로 쪼개기 (불용어 제거)
    return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(d) for d in docs]           # 문서별 단어 개수
        self.len = np.array([len(d) for d in docs], dtype="float32")
        self.avg = self.len.mean()
        df = Counter(w for d in docs for w in set(d))  # 단어가 몇 개 문서에 나오는지
        N = len(docs)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}   # 희귀할수록 큼

    def scores(self, query):
        s = np.zeros(len(self.tf), dtype="float32")
        for w in set(tokenize(query)):
            if w not in self.idf:
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)
                if f:                                  # 단어가 많이 나올수록, 문서가 짧을수록 점수 ↑
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

bm25 = BM25([tokenize(c["meta"]["title"] + " " + c["meta"]["title"] + " " + c["text"]) for c in chunks])

def bm25_search(q, k=10):
    return [chunks[i] for i in np.argsort(-bm25.scores(q))[:k]]

print([c["meta"]["title"] for c in bm25_search("Two-Weapon Fighting", 3)])

## 7. 최종 검색 함수 `search_final` (임베딩 3갈래 + 키워드 2갈래 → 가중 RRF)
RRF: 각 검색의 순위를 `비중/(60+순위)` 점수로 바꿔 더해요. 임베딩 3표(원문 질문, 번역①, 번역②)는 비중 1.0, 키워드 2표(번역①, ②)는 비중 0.5예요.

In [ ]:
# Ex07. 검색: 질문 하나 → 청크 5개
def search_two(texts, k=5):                           # 문장 목록 → 청크 k개 (여러 문장이면 최고 점수로 합침)
    v = np.array(embed_batch(texts), dtype="float32")
    faiss.normalize_L2(v)
    sc, ids = index.search(v, k)
    best = {}
    for srow, irow in zip(sc, ids):
        for s, i in zip(srow, irow):
            best[i] = max(best.get(i, -1.0), float(s))
    return [chunks[i] for i in sorted(best, key=best.get, reverse=True)[:k]]

def fuse_w(lists, weights, k=5, c0=60):               # 여러 검색 결과를 비중을 줘서 합치기
    score, byid = {}, {}
    for lst, w in zip(lists, weights):
        for r, c in enumerate(lst, 1):                # 순위가 높을수록 큰 점수
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            byid[c["id"]] = c
    return [byid[i] for i in sorted(score, key=score.get, reverse=True)[:k]]

def search_final(q, k=5):
    en1, en2 = rewrite_query(q), rewrite_query2c(q)   # 번역①(일반), 번역②(용어 선택지)
    lists = [search_two([t], 10) for t in (q, en1, en2)] + [bm25_search(en1, 10), bm25_search(en2, 10)]
    return fuse_w(lists, [1, 1, 1, 0.5, 0.5], k), [q, en1, en2]

## 8. 번역·검색 캐시 (개발셋을 빠르게 + 런타임이 초기화돼도 유지)

**왜 필요해요?** T4에서는 `14B ↔ bge-m3`를 번갈아 부르면 모델이 메모리에서 교체돼서 질문당 18~20초가 걸려요. 번역과 검색 결과를 드라이브에 저장해 두면 개발셋은 질문당 약 6초로 줄어요.

**순서가 중요해요 (캐시 파일 이름 `cache_dev_14b.json`)**
1. 아래 첫 셀: 캐시 장치 설치 + 파일이 있으면 불러오기
2. 파일이 없거나 새 질문이 생기면 → 둘째 셀 `build_cache(...)` (번역만 먼저 → 검색만 나중에)
3. 셋째 셀: 개발셋 검색 점수 확인 후 저장

옛 캐시(`cache_dev.json`, `cache_dev_v1.json`)는 7B 번역 기준이라 쓰지 않아요.

In [ ]:
# Ex08-a. 캐시 장치: 번역·검색 결과를 기억하는 래퍼 + 드라이브 저장/불러오기
CACHE_PATH = f"{DATA}/cache_dev_14b.json"
by_id = {c["id"]: c for c in chunks}
_tr_cache, _sf_cache = {}, {}

if "_orig_q1" not in globals():                       # 원래 함수를 한 번만 보관 (다시 실행해도 이중으로 안 감싸요)
    _orig_q1, _orig_q2, _orig_sf = rewrite_query, rewrite_query2c, search_final

def rewrite_query(q):
    if ("1", q) not in _tr_cache: _tr_cache[("1", q)] = _orig_q1(q)
    return _tr_cache[("1", q)]
def rewrite_query2c(q):
    if ("2", q) not in _tr_cache: _tr_cache[("2", q)] = _orig_q2(q)
    return _tr_cache[("2", q)]
def search_final(q, k=5):
    if (q, k) not in _sf_cache: _sf_cache[(q, k)] = _orig_sf(q, k)
    return _sf_cache[(q, k)]

def save_cache():
    json.dump({
        "tr": {f"{k[0]}|{k[1]}": v for k, v in _tr_cache.items()},                 # "1|질문" 형태
        "sf": {f"{k[0]}|{k[1]}": {"ids": [c["id"] for c in v[0]], "texts": v[1]}   # "질문|5" 형태, 청크는 id만
               for k, v in _sf_cache.items()},
    }, open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("저장:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")

def load_cache():
    if not os.path.exists(CACHE_PATH):
        print("캐시 파일 없음 → 아래 build_cache(dev)로 만들어요")
        return
    d = json.load(open(CACHE_PATH, encoding="utf-8"))
    for k, v in d["tr"].items():
        _tr_cache[tuple(k.split("|", 1))] = v
    for k, v in d["sf"].items():
        q, kk = k.rsplit("|", 1)
        _sf_cache[(q, int(kk))] = ([by_id[i] for i in v["ids"]], v["texts"])
    print("불러옴:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")

def unload(model):                                    # 쓴 직후 모델을 메모리에서 내려요 (T4 메모리 확보)
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0})

dev = golden + golden_new                             # 개발셋 38문항
load_cache()

In [ ]:
# Ex08-b. 캐시 만들기: ① 번역만 전부(7B) → ② 검색만 전부(bge-m3). 이미 캐시된 질문은 건너뛰어요
def build_cache(items):
    qs = [g["q"] for g in items]
    todo_tr = [q for q in qs if ("1", q) not in _tr_cache or ("2", q) not in _tr_cache]
    if todo_tr:
        _orig_q1(todo_tr[0])                          # 워밍업: 모델 첫 호출 흔들림을 흘려보냄 (결과 버림)
        for q in tqdm(todo_tr, desc="번역(14B)"):
            rewrite_query(q); rewrite_query2c(q)
        unload(TRANSLATE_MODEL)
    todo_sf = [q for q in qs if (q, 5) not in _sf_cache]
    for q in tqdm(todo_sf, desc="검색(bge-m3)"):
        search_final(q)
    if todo_sf:
        unload(EMBED_MODEL)
    print(f"완료. 번역 새로 {len(todo_tr)}개, 검색 새로 {len(todo_sf)}개 → 캐시 번역 {len(_tr_cache)} / 검색 {len(_sf_cache)}")

build_cache(dev + golden_ko)                          # 개발 3세트 (영어 25 + new 13 + ko 28)

In [ ]:
# Ex08-c. 번역을 14B로 바꾼 뒤의 개발셋 검색 점수 (옛 v1 = 7B 번역 기준과 비교만 해요)
V1_DEV = {"영어": (20, 24, 24), "한국어ko": (16, 23, 24), "한국어new": (5, 7, 9)}     # 7B 번역일 때 점수
sets3 = {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}
tot = [0, 0, 0]
for name, items in sets3.items():
    got = counts([rank_in(search_final(g["q"])[0], g)[0] for g in items])
    tot = [x + y for x, y in zip(tot, got)]
    print(f"{name:8s} 14B 번역 {got} | 7B 번역(v1) {V1_DEV[name]}")
print("합계(66):", tuple(tot), "| 7B 번역(v1): (41, 54, 57)")
save_cache()

## 9. 답변 생성 도구 (14B)
근거에 번호를 붙여 **번호를 먼저 고르고** 그 근거에서만 인용하게 해요. 용어표(`TERM_MAP`)는 검색용이 아니라 **답변의 한국어 표기를 통일**하는 용도예요.

In [ ]:
# Ex09-a. 용어표 (한국어 = 룰북 영어 용어) — 답변용
TERM_MAP = {
    # 상태
    "실명": "Blinded", "매료": "Charmed", "귀먹음": "Deafened", "탈진": "Exhaustion",
    "공포": "Frightened", "붙잡힘": "Grappled", "행동 불능": "Incapacitated", "투명": "Invisible",
    "마비": "Paralyzed", "석화": "Petrified", "중독": "Poisoned", "넘어짐": "Prone",
    "포박": "Restrained", "기절": "Stunned", "의식 불명": "Unconscious",
    # 판정·전투
    "이점": "Advantage", "불리점": "Disadvantage", "능력 판정": "Ability Check",
    "공격 굴림": "Attack Roll", "내성 굴림": "Saving Throw", "기회 공격": "Opportunity Attack",
    "반응": "Reaction", "추가 행동": "Bonus Action", "집중": "Concentration",
    "이니셔티브": "Initiative", "저항": "Resistance", "취약": "Vulnerability",
    "엄폐": "Cover", "치명타": "Critical Hit", "숙련": "Proficiency", "전문화": "Expertise",
    "긴 휴식": "Long Rest", "짧은 휴식": "Short Rest",
    # 능력치
    "근력": "Strength", "민첩": "Dexterity", "건강": "Constitution",
    "지능": "Intelligence", "지혜": "Wisdom", "카리스마": "Charisma",
}
TERMS = ", ".join(f"{en}={ko}" for ko, en in TERM_MAP.items())     # 답변용 (영어=한국어)

In [ ]:
# Ex09-a2. 용어표 v1 (룰북 기준으로 한 번 만들고 고정). 결과를 보고 용어를 더 넣지 않아요.
# 처음 한 번만 만들어 드라이브에 저장하고, 그 뒤로는 파일을 그대로 불러와요 (수정 금지, 바꾸려면 v2로).
V1_PATH = f"{DATA}/term_map_v1.json"
if os.path.exists(V1_PATH):
    TERM_V1_NEW = json.load(open(V1_PATH, encoding="utf-8"))
    print("용어표 v1 불러옴:", len(TERM_V1_NEW), "개")
else:
    TERM_V1_NEW = {
        # 피해 종류 13
        "산성": "Acid", "타격": "Bludgeoning", "냉기": "Cold", "화염": "Fire", "역장": "Force",
        "번개": "Lightning", "괴사": "Necrotic", "관통": "Piercing", "독": "Poison", "정신": "Psychic",
        "광휘": "Radiant", "참격": "Slashing", "천둥": "Thunder",
        # 클래스 12
        "야만전사": "Barbarian", "바드": "Bard", "클레릭": "Cleric", "드루이드": "Druid", "파이터": "Fighter",
        "몽크": "Monk", "팔라딘": "Paladin", "레인저": "Ranger", "로그": "Rogue", "소서러": "Sorcerer",
        "워락": "Warlock", "위저드": "Wizard",
        # 마법 학파 8
        "방호": "Abjuration", "소환": "Conjuration", "예지": "Divination", "환혹": "Enchantment",
        "방출": "Evocation", "환영": "Illusion", "강령": "Necromancy", "변환": "Transmutation",
        # 주문 칸
        "주문 칸": "Spell Slot",
    }
    json.dump(TERM_V1_NEW, open(V1_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("⚠️ 파일이 없어 기본 목록(피해 종류·클래스·학파·주문 칸)만 만들었어요. 룰북 용어집 항목은 포함되지 않았어요.")
TERM_MAP.update(TERM_V1_NEW)

def terms_for(text):                                   # 고른 근거에 실제로 나온 용어만 프롬프트에 넣어요
    out = []
    for ko, en in TERM_MAP.items():
        if " " in en:                                  # 여러 단어: 대소문자 무시 + 복수형
            hit = re.search(rf"\b{re.escape(en)}s?\b", text, re.I)
        else:                                          # 한 단어: 대소문자 구분 (문장 속 force, fire 같은 일반 단어 제외)
            hit = re.search(rf"\b{re.escape(en)}\b", text)
        if hit: out.append(f"{en}={ko}")
    return ", ".join(out) or "(없음)"

print("용어 총", len(TERM_MAP), "개 |", terms_for("A Prone creature has Disadvantage on Attack Rolls. Fire damage and Long Rest."))

In [ ]:
# Ex09-b. 호출 함수 + 깨짐/혼합/분수 검사
SYSTEM = ("You are a D&D rules assistant. Always answer in Korean (Hangul) only. "
          "Never use Chinese characters or Chinese words.")
GEN_SEED = 42                                           # 같은 입력 → 같은 출력 (평가용). 챗봇 재시도는 seed를 바꿔요

def ask_llm(prompt, model=None, temperature=0.1, seed=GEN_SEED):      # 설명 생성용
    opts = {"num_ctx": 8192, "temperature": temperature}
    if seed is not None: opts["seed"] = seed                          # seed가 실제로 Ollama에 전달돼야 재현돼요
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": SYSTEM},     # 수업 [3]A Ex13: system 역할
                           {"role": "user", "content": prompt}],
              "stream": False, "options": opts},                      # 수업 [3]C: /api/chat
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

def gen(prompt, model=None, system=SYSTEM):            # 선택·검증·충실성 검사용 (temperature 0 + seed 42)
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False,
              "options": {"num_ctx": 8192, "temperature": 0, "seed": 42}},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

# 수업에 없음: re.compile + 유니코드 범위. "이 글자들 중 하나라도 있으면 깨짐"이라는 검사 패턴이에요.
#  \u4e00-\u9fff 한자 / \u3040-\u30ff 일본어 / \u1ea0-\u1ef9 및 ăâêôơưđ 베트남어 / \ufffd 깨진 글자
BROKEN = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\u1ea0-\u1ef9\u0300-\u036făâêôơưđĐ\ufffd]")
def is_broken(text):
    return bool(BROKEN.search(text))

LEAK = re.compile(r"(^|\n)\s*(user|assistant|system)\s*(\n|:)", re.I)     # 줄 처음에 "user" 같은 말이 새어 나왔는지

def has_chinese(text):                                 # (옛 이름 유지) 중국어 글자 확인
    return bool(re.search(r"[\u4e00-\u9fff]", text))

MIXED = re.compile(r"[가-힣][A-Za-z]")                  # 한글 바로 뒤에 영어 철자가 붙음 (예: "이점Advantage", "스티일")
FRACTION = re.compile(r"\d+분의\s*\d+")                # "3분의 2" — 14B가 Three-Quarters를 뒤집는 오류가 있었어요

def bad_output(text):                                  # 재생성이 필요한 출력인가
    return is_broken(text) or bool(MIXED.search(text)) or bool(FRACTION.search(text))

In [ ]:
# Ex09-c. 답변 프롬프트
RAG_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 규칙을 반드시 지키세요.
1. [근거]는 번호가 붙은 여러 개입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요. 관련 없는 근거는 무시하세요.
2. 고른 근거에 적힌 내용만 사용하세요. 어느 근거에도 없으면 "근거에서 찾지 못했어요"라고만 답하세요.
3. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적으세요: {terms}
4. 근거의 조건, 숫자, 이점/불리점을 빼거나 바꾸지 마세요.
5. 답변은 "근거 번호:", "원문:", "설명:" 세 줄로 쓰세요. 원문은 고른 근거에서 질문에 직접 답하는 문장을 영어 그대로 복사하고, 설명은 한국어로 쓰세요.

### 예시
질문: What does the Dash action do?
근거:
[1] (Torch (1 CP) p.100) A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
[2] (Dash [Action] p.181) When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
답변:
근거 번호: 2
원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
설명: 질주(Dash) 행동을 하면 이번 턴에 속도(Speed)만큼 이동 거리가 늘어납니다.

### 근거
{context}

### 질문
{question}

### 답변"""

## 10. 질문 함수 `rag_answer` / `ask_rules` (검색 → 근거 → 답변)

In [ ]:
# Ex10. 질문 하나 → 검색 → 답변. 챗봇 본체에서 이 함수를 부름
def rag_answer(question, k=5, tries=3):
    found, texts = search_final(question, k)
    context = "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
                        for n, c in enumerate(found, 1))
    prompt = RAG_TEMPLATE.format(terms=TERMS, context=context, question=question)
    for t in range(1, tries + 1):                      # 한자·일본어·베트남어 등이 섞이면 temperature를 올려 다시 생성
        answer = ask_llm(prompt, temperature=0.1 + 0.3 * (t - 1))
        if not is_broken(answer):
            break
    sources = [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found]
    return {"answer": answer, "sources": sources, "queries": texts[1:], "tries": t}

def ask_rules(q, answer=True):                         # 노트북에서 직접 확인하는 용도
    if not answer:
        found, texts = search_final(q)
        print("번역:", texts[1], "|", texts[2])
        print("검색:", [f"{c['meta']['title']} (p.{c['meta']['page']})" for c in found])
        return
    r = rag_answer(q)
    print("번역:", r["queries"][0], "|", r["queries"][1])
    print("검색:", r["sources"])
    print("\n" + r["answer"])

In [ ]:
# Ex11. 검색 결과를 텍스트 파일로 저장해서 눈으로 확인 (✅ = 정답 제목과 일치)
def save_retrieval_check(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for n, g in enumerate(items, 1):
            found, texts = search_final(g["q"])
            f.write(f"########## Q{n}. {g['q']}\n번역: {texts[1]} | {texts[2]}\n정답 제목: {g['want']}\n\n")
            for r, c in enumerate(found, 1):
                ok = any(is_hit(c["meta"]["title"], w) for w in wants_of(g))
                f.write(f"--- {r}등 {'✅정답' if ok else ''} | {c['meta']['title']} | p.{c['meta']['page']} | {c['id']}\n")
                f.write(c["text"][:400].replace("\n", " ") + "\n\n")
            f.write("\n")

# CHECK_PATH = f"{DATA}/retrieval_check.txt"
# save_retrieval_check(golden_new, CHECK_PATH)
# files.download(CHECK_PATH)

## 11. 개발셋 검색 평가
캐시 덕분에 번역·임베딩 호출 없이 바로 나와요. 기준(v1): 영어 (20,24,24), ko (16,23,24), new (5,7,9).

In [ ]:
# Ex12. 개발 3세트 검색 점수 (1등 / 3등 안 / 5등 안)
fn = lambda q: search_final(q)[0]
tot = [0, 0, 0]
for name, items in sets3.items():
    c = evaluate(fn, items)
    tot = [a + b for a, b in zip(tot, c)]
    print(f"{name} ({len(items)}문항): {c}")
print("합계:", tuple(tot), "/ 66")        # v1: (41, 54, 57)

## 12. 설명 품질 실험 도구 (개발셋 38문항, 정답 청크로 생성)

검색이 틀린 경우와 **설명이 틀린 경우를 분리**하려고, 정답 청크를 직접 넣어서 설명만 평가해요.
문항마다 기준을 손으로 쓰지 않고(post-hoc, 순환 위험) **자동 지표**만 써요:
깨짐 / 대화턴 누출 / 원문 복사 / 숫자 보존 / 용어 보존 / 용어 불일치.

**결정 사항**: 생성(설명+근거 선택)은 14B + 현행 `RAG_TEMPLATE`. 좁힌 프롬프트(`NARROW_TEMPLATE`)는 이득이 없어서 채택하지 않았어요 (비교용으로만 남겨 둠).

In [ ]:
# Ex13-a. 개발셋 정답 청크 매칭
def gold_chunks(g, limit=3):       # want 제목과 일치하는 청크를 찾고, 질문당 최대 limit개만 사용
    hits = [c for c in chunks
            if any(is_hit(c["meta"]["title"], w) for w in wants_of(g))]
    return hits[:limit]

for g in dev:
    g["gold"] = gold_chunks(g)

print(len(dev), "문항")
print("정답 청크 0개인 문항:", [g["q"] for g in dev if not g["gold"]])
print("청크 수 분포:", [len(g["gold"]) for g in dev])

In [ ]:
# Ex13-b. 평가 도구
def build_ctx(g):                                   # 정답 청크를 번호 붙여 근거로 만들기
    return "\n".join(f"[{n}] ({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
                     for n, c in enumerate(g["gold"], 1))

def expl_part(answer):                              # "설명:" 뒤만 평가 (근거 번호·원문 줄은 제외)
    m = re.search(r"설명\s*[:：]\s*(.*)", answer, flags=re.S)
    return m.group(1).strip() if m else answer

NUM = re.compile(r"\d+(?:d\d+)?", re.I)            # 숫자 또는 주사위 표기(2d6)
def numbers(text):
    return [n.lower() for n in NUM.findall(text)]

def src_text(g):
    return " ".join(c["text"] for c in g["gold"])

# 영어 숫자 단어 → 숫자 (주의: "Three-Quarters" 같은 분수를 숫자로 풀어 버려서 "3분의 4" 같은 뒤집힘 오류를 놓칠 수 있어요)
W2N = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7",
       "eight": "8", "nine": "9", "ten": "10", "half": "1 2", "quarter": "1 4", "quarters": "4"}
W2N.update({"twice": "2", "double": "2", "doubled": "2"})      # 두 배 표현도 숫자 2로 봐요

def src_numbers(g):
    s = src_text(g).lower()
    extra = [d for w in re.findall(r"[a-z]+", s) if w in W2N for d in W2N[w].split()]
    return set(numbers(s)) | set(extra)

def score(rows, items=None):                         # 간단 요약: 깨짐 / 누출 / 숫자 충실도
    items = items or dev
    broken, leak, tot, ok, bad_q = [], [], 0, 0, []
    for r, g in zip(rows, items):
        e = expl_part(r["answer"])
        if is_broken(e): broken.append(g["q"])
        if LEAK.search(e): leak.append(g["q"])
        src, nums = src_numbers(g), numbers(e)
        good = sum(n in src for n in nums); tot += len(nums); ok += good
        if good < len(nums): bad_q.append((g["q"], [n for n in nums if n not in src]))
    print(f"깨짐 {len(broken)}/{len(rows)} | 대화턴 누출 {len(leak)} | 숫자 충실도 {ok}/{tot} | 근거에 없는 숫자 문항 {len(bad_q)}")
    return broken, bad_q

norm_txt = lambda s: re.sub(r"\s+", " ", re.sub(r"[\"'“”‘’]", "", s)).strip().lower()
TERM_REV = {en.lower(): ko for ko, en in TERM_MAP.items()}        # 영어 → 한국어

def eval_expl(name, rows, items=None):               # 자세한 자동 채점 (문항별 기준을 손으로 쓰지 않아요)
    items = items or dev
    n_orig = copy_ok = nums_t = nums_ok = term_t = term_ok = 0
    mism = []
    for r, g in zip(rows, items):
        ans = r["answer"]
        orig = " ".join(re.findall(r"원문\s*[:：]\s*(.+)", ans))      # 모델이 고른 영어 문장
        e = expl_part(ans)
        if orig:
            n_orig += 1
            copy_ok += norm_txt(orig) in norm_txt(src_text(g))        # 근거 안에 실제로 있는 문장?
        o_nums, e_nums = numbers(orig), set(numbers(e))
        nums_t += len(o_nums); nums_ok += sum(n in e_nums for n in o_nums)
        for ko, en in TERM_MAP.items():                               # 원문의 용어표 단어 → 한국어가 설명에 있나
            if re.search(rf"\b{re.escape(en)}\b", orig, re.I):
                term_t += 1; term_ok += ko in e
        for m in re.finditer(r"\(([A-Za-z][A-Za-z \-]+)\)", e):       # 한글(English) 쌍이 용어표와 맞나
            ko = TERM_REV.get(m.group(1).strip().lower())
            if ko and ko not in e[max(0, m.start() - 20):m.start()]:
                mism.append((g["q"][:25], ko, m.group(1)))
    broken = sum(is_broken(expl_part(r["answer"])) for r in rows)
    leak = sum(bool(LEAK.search(expl_part(r["answer"]))) for r in rows)
    print(f"{name:12s} 깨짐 {broken} | 누출 {leak} | 원문 복사 {copy_ok}/{n_orig} | 숫자 보존 {nums_ok}/{nums_t} | "
          f"용어 보존 {term_ok}/{term_t} | 용어 불일치 {len(mism)}")
    return mism

In [ ]:
# Ex13-c. 좁힌 프롬프트 (비교용, 채택 안 함) + 변형 실행기
NARROW_TEMPLATE = """당신은 D&D 규칙을 한국어로 옮기는 번역가입니다. 아래 규칙을 반드시 지키세요.
1. [근거]는 번호가 붙은 영어 규칙입니다. 질문에 가장 직접 답하는 근거 하나를 고르세요.
2. 고른 근거에 적힌 내용만 한국어로 옮기거나 요약하세요. 근거에 없는 내용, 추측, 일반 상식을 덧붙이지 마세요.
3. 숫자, 주사위 표기(예: 2d6), 거리·시간 단위, DC, +2 같은 보너스는 근거에 적힌 그대로 쓰세요. 바꾸거나 계산하지 마세요.
4. 규칙 용어는 이 표의 한국어를 쓰고 영어를 괄호로 함께 적으세요. 표에 없는 용어는 영어 그대로 두세요: {terms}
5. 한글, 영어 알파벳, 숫자만 사용하세요. 한자, 일본어, 베트남어는 쓰지 마세요.
6. 답변은 "근거 번호:", "원문:", "설명:" 세 줄로 쓰세요. 원문은 고른 근거에서 질문에 직접 답하는 문장을 영어 그대로 복사하세요.

### 예시
질문: What does the Dash action do?
근거:
[1] (Torch (1 CP) p.100) A Torch burns for 1 hour, casting Bright Light in a 20-foot radius and Dim Light for an additional 20 feet.
[2] (Dash [Action] p.181) When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
답변:
근거 번호: 2
원문: When you take the Dash action, you gain extra movement for the current turn. The increase equals your Speed after applying any modifiers.
설명: 질주(Dash) 행동을 하면 이번 턴에 이동 거리가 늘어납니다. 늘어나는 양은 수정치를 적용한 속도(Speed)와 같습니다.

### 근거
{context}

### 질문
{question}

### 답변"""

def run_variant(name, model, template, items=None):
    items = items or dev
    path = f"{DATA}/expl_{name}.json"
    if os.path.exists(path):
        print("캐시 사용:", name)
        return json.load(open(path, encoding="utf-8"))["rows"]
    g0 = items[0]
    gen(template.format(terms=TERMS, context=build_ctx(g0), question=g0["q"]), model)   # 워밍업: 실제 프롬프트로 (결과 버림)
    rows, t0 = [], time.time()
    for g in tqdm(items, desc=name):
        prompt = template.format(terms=TERMS, context=build_ctx(g), question=g["q"])
        rows.append({"q": g["q"], "answer": gen(prompt, model)})
    sec = (time.time() - t0) / len(items)
    json.dump({"model": model, "sec_per_q": sec, "rows": rows},
              open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"{name}: 질문당 {sec:.1f}초")
    return rows

In [ ]:
# Ex13-d. 4가지 변형 결과 비교 (저장된 파일이 있으면 바로 불러와요. 없으면 새로 생성)
res = {}
for _k, _n, _t in (("A_7B_현행", "A_7B_current", RAG_TEMPLATE), ("B_7B_좁힘", "B_7B_narrow", NARROW_TEMPLATE)):
    if os.path.exists(f"{DATA}/expl_{_n}.json"):        # 7B는 예전에 저장한 결과가 있을 때만 비교에 넣어요 (새로 안 돌려요)
        res[_k] = run_variant(_n, "qwen2.5:7b", _t)
res["C_14B_현행"] = run_variant("C_14B_current", "qwen2.5:14b", RAG_TEMPLATE)    # ← 채택
res["D_14B_좁힘"] = run_variant("D_14B_narrow",  "qwen2.5:14b", NARROW_TEMPLATE)

for k, rows in res.items():
    print(k); score(rows)
print()
mism_all = {k: eval_expl(k, rows) for k, rows in res.items()}
for k, m in mism_all.items():
    print(k, m[:6])

In [ ]:
# Ex13-e. 14B 현행(C) 답변에서 의심 문항 직접 보기 (깨짐 / 숫자가 근거에 없음 / 용어 불일치)
rows = res["C_14B_현행"]
for r, g in zip(rows, dev):
    e = expl_part(r["answer"])
    bad = sorted(set(BROKEN.findall(e)))
    extra = [n for n in numbers(e) if n not in src_numbers(g)]
    if bad or extra:
        print(f"[깨진 글자 {bad} | 근거에 없는 숫자 {extra}] {g['q']}\n   {e[:300]}\n")

## 13. 근거 선택 실험: 한 번에 하기 vs 2단계

- 한 번에 하기(현행): 14B가 번호·원문·설명을 한 번에 → 개발셋 46/56 (82%), 7B는 44/57. "1등 고정" 기준선은 약 69%.
- **2단계**: ① 후보 5개의 제목+앞 300자만 보여 주고 **번호만 고르게** → ② 고른 근거 하나로 설명. 코드가 원문+페이지를 직접 붙일 수 있어요.
- 주의: 번역을 14B로 바꿨기 때문에 검색 결과가 옛 v1(7B 번역)과 달라질 수 있어요. 8번 구간 점수를 먼저 확인하세요.

In [ ]:
# Ex14-a. 2단계 방식의 1단계: 후보 5개 중 번호만 고르기 (개발셋 평가)
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""

def select_idx(q, found, model=None):
    cands = "\n".join(f"[{n}] {c['meta']['title']}: {c['text'][:300].replace(chr(10), ' ')}"
                      for n, c in enumerate(found, 1))
    out = gen(SELECT_TEMPLATE.format(question=q, cands=cands), model)      # gen: temperature 0, seed 42
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    return n if 1 <= n <= len(found) else 1                                # 번호를 못 읽으면 1등으로

def eval_select(items, model=None):
    H = C = R1 = 0
    wrong = []
    for g in tqdm(items, leave=False):
        found = search_final(g["q"])[0]                                   # 캐시된 검색 결과
        titles = [c["meta"]["title"] for c in found]
        hit = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
        if not hit:
            continue                                                       # 검색이 못 찾은 문항은 제외
        H += 1; R1 += (hit == 1)
        n = select_idx(g["q"], found, model)
        ok = any(is_hit_loose(titles[n - 1], w) for w in wants_of(g))      # 선택 평가에서만 단수/복수 허용
        C += ok
        if not ok:
            wrong.append((g["q"][:35], g["want"], f"정답 {hit}등", f"고른 것 {n}: {titles[n - 1]}"))
    return H, C, R1, wrong

In [ ]:
# Ex14-b. 점검: 검색 점수 + 선택 호출 속도
for name, items in sets3.items():
    ranks = [rank_in(search_final(g["q"])[0], g)[0] for g in items]
    print(name, "1등/3등/5등 안:", counts(ranks), "| 후보 개수:", {len(search_final(g["q"])[0]) for g in items})
print("7B 번역(옛 v1) 기준:", V1_DEV)

g = golden_new[0]; found = search_final(g["q"])[0]
select_idx(g["q"], found)                              # 워밍업 (14B를 올리는 시간 제외)
t0 = time.time()
for _ in range(3):
    select_idx(g["q"], found)
print("선택 호출 1번:", round((time.time() - t0) / 3, 1), "초")      # 1초 안팎이면 정상

In [ ]:
# Ex14-c. 2단계 선택 평가
base14 = {"영어": "20/24", "한국어ko": "18/23", "한국어new": "8/9"}   # 한 번에 하기(14B) 이전 측정
tot = [0, 0, 0]
for name, items in sets3.items():
    H, C, R1, wrong = eval_select(items)
    tot[0] += H; tot[1] += C; tot[2] += R1
    print(f"{name}: 검색 적중 {H} | 2단계 선택 정답 {C}/{H} | (1등 고정 {R1}/{H}, 한 번에 하기 14B {base14[name]})")
    for w in wrong:
        print("   ", w)
print(f"합계: 2단계 {tot[1]}/{tot[0]} | 1등 고정 {tot[2]}/{tot[0]}")

In [ ]:
# Ex14-d. 한 번에 하기 방식으로 근거를 제대로 고르는지 (옛 평가, 14B 현행 rag_answer 사용)
def pick_no(answer):
    m = re.search(r"근거 번호\s*[:：]\s*(\d+)", answer)
    return int(m.group(1)) if m else None

def eval_pick(items):
    rows = []
    for g in tqdm(items):
        r = rag_answer(g["q"])
        titles = [re.sub(r"\s*\(p\.\d+\)$", "", s) for s in r["sources"]]
        hit_rank = next((i + 1 for i, t in enumerate(titles) if any(is_hit(t, w) for w in wants_of(g))), None)
        n = pick_no(r["answer"])
        ok = n is not None and 1 <= n <= len(titles) and any(is_hit(titles[n - 1], w) for w in wants_of(g))
        rows.append({"q": g["q"], "want": g["want"], "hit_rank": hit_rank, "picked": n,
                     "picked_title": titles[n - 1] if n and 1 <= n <= len(titles) else None, "ok": ok})
    return rows

# 오래 걸려요(질문마다 14B 생성). 필요할 때만 주석을 풀어 실행하세요.
# all_rows = {name: eval_pick(items) for name, items in sets3.items()}
# for name, rows in all_rows.items():
#     H = [x for x in rows if x["hit_rank"]]; C = [x for x in H if x["ok"]]
#     print(f"{name}: 검색 적중 {len(H)}/{len(rows)} | 근거 선택 정답 {len(C)}/{len(H)}")

In [ ]:
# Ex15. 속도 점검: 캐시 뒤 질문당 시간 (6초 안팎이어야 해요. 전에는 18~20초)
rag_answer(golden_new[0]["q"])                         # 워밍업: 14B 올리는 시간은 빼고 재요
t0 = time.time()
for g in golden_new[1:4]:
    r = rag_answer(g["q"])
print("질문당", round((time.time() - t0) / 3, 1), "초")
!ollama ps

## 15. 2단계 답변 파이프라인 (선택 → 검증/거절 → 설명 → 자동 검사)

```
질문 → search_final(5개) → select_idx(번호) → verify_chunk(예/아니오)
        └ 아니오 → 거절(후보 제목만) ─┐
        └ 예 → rag_answer2(원문+설명) → check_answer2(규칙 검사) → faithful(근거에 없는 내용?)
                                      └ 걸리면: 개발 모드 = 숨기고 근거 본문 표시 / 챗봇 모드 = 다른 seed로 재시도 → 그래도 걸리면 "확실하지 않아요"
```
- 원문·제목·페이지는 **모델이 아니라 코드가** 붙여요 (원문이 근거에 실제로 있는지 코드가 확인).
- 설명 단계는 선택된 **근거 하나만** 봐요. 용어표도 그 근거에 나온 용어만 넣어요 (`terms_for`).
- 정책 A: 검증이 "아니오"면 거절해요 → 거절이 얼마나 늘어나는지는 Ex16-b, 걸림 비율은 Ex16-d에서 재요.
- 개발셋(38문항)은 디버깅·회귀 확인용이에요 (이미 오염됨). 점수를 최종 성능으로 보고하지 않아요.

In [ ]:
# Ex15-b. 2단계 답변: 근거 하나만 보고 "원문 + 설명" (원문·제목·페이지는 코드가 확인해서 붙임)
EXPLAIN_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 [근거]만 보고 질문에 답하세요.
1. 근거에 적힌 내용만 사용하세요. 근거에 없는 내용, 추측, 일반 상식을 덧붙이지 마세요.
2. 용어는 이 표를 따르고 같은 용어는 항상 같은 한국어로 쓰세요. 한국어 뒤에 영어를 괄호로 적으세요: {terms}
3. '이점'은 Advantage에만 쓰세요. bonus는 보너스, roll은 굴림, check는 판정이라고 쓰세요.
4. 숫자, 분수, 조건, 예외를 빼거나 바꾸지 마세요.
5. 답변은 두 줄로 쓰세요.
원문: 근거에서 질문에 직접 답하는 문장 1~3개를 영어 그대로, 한 줄로 복사
설명: 근거의 내용만 한국어로 풀어 쓴 설명

### 근거
{context}

### 질문
{question}

### 답변"""

def rag_answer2(question, k=5, tries=3, temp0=0.1, seed0=40, pre=None):
    found, texts = pre[:2] if pre else search_final(question, k)
    n = pre[2] if pre else select_idx(question, found)               # 1단계: 번호만 고르기
    c = found[n - 1]
    ctx = f"({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:2000]}"
    prompt = EXPLAIN_TEMPLATE.format(terms=terms_for(c["text"]), context=ctx, question=question)
    for t in range(1, tries + 1):                                    # 깨짐·혼합 철자·분수가 나오면 seed/온도를 바꿔 재생성
        out = ask_llm(prompt, temperature=temp0 + 0.3 * (t - 1), seed=seed0 + t)
        if not bad_output(out):
            break
    m = re.search(r"원문\s*[:：]\s*(.+)", out)
    orig = m.group(1).strip() if m else ""
    verified = bool(orig) and norm_txt(orig) in norm_txt(c["text"])   # 원문이 근거 안에 실제로 있나 (코드가 확인)
    return {"title": c["meta"]["title"], "page": c["meta"]["page"], "text": c["text"], "picked": n,
            "orig_ok": verified,
            "orig": orig if verified else c["text"].split("\n", 1)[-1][:300],   # 검증 실패 시 근거 앞부분으로 대체
            "path": c["text"].split("\n", 1)[0], "explain": expl_part(out),
            "queries": texts[1:], "tries": t}

SENT_END = re.compile(r'[.!?)\]"\u201d]\s*$')

def chunk_numbers(text):                                              # 근거 안의 숫자 (영어 숫자 단어 포함)
    s = text.lower()
    extra = [d for w in re.findall(r"[a-z]+", s) if w in W2N for d in W2N[w].split()]
    return set(numbers(s)) | set(extra)

def check_answer2(r):                                                 # 규칙 기반 자동 검사 (LLM 호출 없음)
    e = r["explain"]
    missing = []
    if r["orig_ok"]:                                                  # 용어 누락은 참고용(info only)
        for ko, en in TERM_MAP.items():
            pat = rf"\b{re.escape(en)}s?\b" if " " in en else rf"\b{re.escape(en)}\b"
            if re.search(pat, r["orig"], re.I if " " in en else 0) and ko not in e:
                missing.append(ko)
    src_nums = chunk_numbers(r["text"])
    return {"원문_검증": r["orig_ok"],
            "원문_완결": bool(SENT_END.search(r["orig"].strip())) if r["orig_ok"] else False,
            "근거에_없는_숫자": [n for n in numbers(e) if n not in src_nums],
            "용어_누락": missing,
            "혼합_철자": MIXED.findall(e),
            "분수": FRACTION.findall(e)}

In [ ]:
# Ex15-d. 개발셋 38문항 기준선: 2단계 답변을 전부 돌려 자동 검사 결과 보기 (회귀 확인용, 이 점수를 성능으로 보고하지 않아요)
gen("안녕")                                                            # 워밍업 (모델 올리는 시간 제외)
base_rows, base_chk = [], []
for g in tqdm(dev):
    r = rag_answer2(g["q"], temp0=0.0)
    base_rows.append(r); base_chk.append(check_answer2(r))

N = len(dev)
sel_ok = sum(any(is_hit_loose(r["title"], w) for w in wants_of(g)) for r, g in zip(base_rows, dev))
print(f"선택 정답(느슨한 제목) {sel_ok}/{N} | 원문 검증 {sum(c['원문_검증'] for c in base_chk)}/{N} | "
      f"원문 완결 {sum(c['원문_완결'] for c in base_chk)}/{N}")
print(f"깨짐 {sum(is_broken(r['explain']) for r in base_rows)} | 혼합 철자 {sum(bool(c['혼합_철자']) for c in base_chk)} | "
      f"분수 {sum(bool(c['분수']) for c in base_chk)} | 근거에 없는 숫자 {sum(bool(c['근거에_없는_숫자']) for c in base_chk)} | "
      f"(참고) 용어 누락 문항 {sum(bool(c['용어_누락']) for c in base_chk)}")
for r, c, g in zip(base_rows, base_chk, dev):
    flags = [k for k in ("근거에_없는_숫자", "혼합_철자", "분수") if c[k]] + ([] if c["원문_검증"] else ["원문_검증"])
    if flags: print("  -", g["q"][:30], flags)

In [ ]:
# Ex16-a. 근거 검증 · 충실성 검사 · 최종 함수 `answer_final` / `show_final`
VERIFY_TEMPLATE = """다음은 D&D 규칙 질문과 근거 하나입니다. 이 근거가 질문이 묻는 규칙을 직접 설명하고 있나요?
- 이름만 비슷하거나 예시로 지나가듯 언급될 뿐이면 "아니오"입니다.
- "예" 또는 "아니오" 한 단어만 출력하세요.

### 질문
{question}

### 근거
{title}: {text}

### 답"""

def verify_chunk(q, c):                                # 고른 근거가 정말 질문에 답하나? (아니오 → 거절)
    out = gen(VERIFY_TEMPLATE.format(question=q, title=c["meta"]["title"], text=c["text"][:1500]))
    return out.strip().startswith("예")

FAITH_TEMPLATE = """다음 [근거]와 [설명]을 비교하세요. [설명]에 [근거]에 없는 내용(추가된 조건, 숫자, 능력, 일반 상식)이나
[근거]와 뜻이 달라진 부분이 있나요?
- 없으면 "없음"만 출력하세요.
- 있으면 "있음: " 뒤에 해당 부분을 짧게 적으세요.

### 근거
{context}

### 설명
{explain}

### 판정"""

def faithful(r):                                       # 설명이 근거에 충실한가 (LLM 검사, 수업에 없음: LLM-as-judge 방식)
    out = gen(FAITH_TEMPLATE.format(context=r["text"][:2000], explain=r["explain"]))
    return out.strip().startswith("없음")

DEV_VIEW = True        # True: 개발 모드(원문·페이지·근거 본문 표시)  /  False: 챗봇 모드(설명만, 걸리면 재시도 후 거절)

def warns_of(r, chk, check_faith=True):                # 걸린 이유 목록. 값싼 규칙 검사부터 하고, 통과하면 LLM 충실성 검사
    w = []
    if not chk["원문_검증"]: w.append("원문 검증 실패")
    elif not chk["원문_완결"]: w.append("원문이 문장 중간에서 끊김")
    if chk["근거에_없는_숫자"]: w.append(f"근거에 없는 숫자 {chk['근거에_없는_숫자']}")
    if chk["분수"]: w.append(f"분수 표현 {chk['분수']}")
    if chk["혼합_철자"]: w.append(f"혼합 철자 {chk['혼합_철자']}")
    if is_broken(r["explain"]): w.append("깨진 글자")
    if not w and check_faith and not faithful(r): w.append("근거에 없는 내용")
    return w

def answer_final(question, attempts=1):
    found, texts = search_final(question)
    n = select_idx(question, found)
    c = found[n - 1]
    if not verify_chunk(question, c):                  # 정책 A: 검증 "아니오" → 거절
        return {"status": "refuse", "candidates": [(x["meta"]["title"], x["meta"]["page"]) for x in found[:3]],
                "first_flagged": False, "attempts_used": 0, "warn": []}
    first_flagged, r, warn = None, None, []
    for a in range(attempts):                          # 챗봇 모드는 attempts=3: 걸리면 다른 seed/온도로 다시 설명
        r = rag_answer2(question, temp0=0.0 if a == 0 else 0.3, seed0=40 + 10 * a, pre=(found, texts, n))
        warn = warns_of(r, check_answer2(r))
        if first_flagged is None: first_flagged = bool(warn)
        if not warn: break
    return {"status": "hidden" if warn else "ok", "warn": warn, "first_flagged": first_flagged, "attempts_used": a + 1,
            "title": r["title"], "page": r["page"], "path": r["path"], "orig": r["orig"],
            "body": r["text"].split("\n", 1)[-1][:600], "explain": r["explain"]}

def show_final(question, dev_view=None):
    dev_view = DEV_VIEW if dev_view is None else dev_view
    r = answer_final(question, attempts=1 if dev_view else 3)
    if r["status"] == "refuse":
        print("🙅 이 질문에 맞는 규칙 근거를 확실히 찾지 못했어요." + (f" 후보: {r['candidates']}" if dev_view else " 질문을 바꿔 보시겠어요?"))
    elif r["status"] == "hidden":
        if dev_view:
            print(f"⚠️ 설명을 숨겼어요 {r['warn']}\n출처: {r['title']} p.{r['page']} ({r['path']})\n[근거 본문] {r['body']}")
        else:
            print("확실하지 않아요. 질문을 조금 바꿔서 다시 물어봐 주세요.")
    else:
        if dev_view:
            print(f"출처: {r['title']} p.{r['page']} ({r['path']})\n원문: {r['orig']}\n설명: {r['explain']}")
        else:
            print(r["explain"])
    return r

In [ ]:
# Ex16-b. 검증(verify) 평가: 선택이 맞았을 때 "예", 틀렸을 때 "아니오"가 나오는지 / 검색이 실패한 질문은 거절되는지
from collections import Counter
tab, miss_q = Counter(), []
for g in tqdm(dev):
    found = search_final(g["q"])[0]
    titles = [c["meta"]["title"] for c in found]
    hit = any(is_hit(t, w) for t in titles for w in wants_of(g))
    n = select_idx(g["q"], found)
    v = verify_chunk(g["q"], found[n - 1])
    if not hit:
        tab["검색 실패 → 거절(좋음)" if not v else "검색 실패 → 통과(못 거름)"] += 1
    else:
        ok = any(is_hit_loose(titles[n - 1], w) for w in wants_of(g))
        key = ("선택 맞음" if ok else "선택 틀림") + (" → 통과" if v else " → 거절")
        tab[key] += 1
        if ok and not v: miss_q.append(g["q"][:35])
for k, v in sorted(tab.items()): print(f"{k:24s} {v}")
print("잘못 거절한 문항(선택은 맞았는데 거절):", miss_q)

In [ ]:
# Ex16-c. 충실성 검사(faithful) 평가: Ex15-d 답변 중 "근거에 없는 내용"으로 걸리는 것 (눈으로 확인용)
flag_f = []
for r, c, g in tqdm(list(zip(base_rows, base_chk, dev))):
    if c["원문_검증"] and not faithful(r):
        flag_f.append((g["q"][:30], r["explain"][:120]))
print(f"충실성 검사에 걸린 문항 {len(flag_f)}/{len(dev)}")
for q, e in flag_f: print(" -", q, "|", e)

In [ ]:
# Ex16-d. 걸림 비율: 챗봇 모드(재시도 3번)로 개발셋 38문항을 돌렸을 때 얼마나 거절·숨김이 생기나
gen("안녕")
stat, why, t0 = Counter(), Counter(), time.time()
for g in tqdm(dev):
    a = answer_final(g["q"], attempts=3)
    stat[a["status"]] += 1
    if a["first_flagged"]: stat["첫 시도에 걸림"] += 1
    if a["first_flagged"] and a["status"] == "ok": stat["재시도로 살아남"] += 1
    for w in a["warn"]: why[w.split(" [")[0].split(" (")[0]] += 1
N = len(dev)
print(f"{N}문항 | 질문당 {(time.time()-t0)/N:.1f}초")
print(f"정상 답변 {stat['ok']} | 거절(근거 검증 아니오) {stat['refuse']} | 끝내 숨김 {stat['hidden']}")
print(f"첫 시도에 걸림 {stat['첫 시도에 걸림']} → 재시도로 살아남 {stat['재시도로 살아남']}")
print("끝내 숨긴 이유:", dict(why))
print(f"챗봇이 \"확실하지 않아요\"라고 하는 비율: {(stat['refuse'] + stat['hidden'])}/{N}")

### 질문 해 보기 (여기만 바꿔서 다시 실행)
`DEV_VIEW = True`면 원문·페이지·근거 본문까지, `False`면 챗봇처럼 설명만 보여 줘요. 캐시에 없는 새 질문은 모델 교체 때문에 첫 질문이 20초쯤 걸릴 수 있어요.

In [ ]:
question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"     # ← 여기만 바꾸세요
_ = show_final(question)                                    # show_final(question, dev_view=False) 로 챗봇 화면 확인

## 참고: 재순위(rerank) 실험 — 채택하지 않음
`bge-reranker-v2-m3`로 후보 20개를 다시 정렬해 봤지만 1등을 뺏기는 질문이 생겨서 쓰지 않아요. 기록용으로만 남겨 둬요 (실행하려면 주석 해제).

In [ ]:
# Ex16. 재순위 실험 (참고용, 기본은 실행 안 함). 후보 20개 -> 재순위 -> 5개
# !pip install -q sentence-transformers
# from sentence_transformers import CrossEncoder
# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")
#
# def rerank_fn(use="ko", pool_k=20, k=5):
#     def fn(q):
#         pool, texts = search_final(q, pool_k)       # texts = [원문, 번역1, 번역2]
#         query = q if use == "ko" else texts[1]
#         scores = reranker.predict([(query, c["text"][:1500]) for c in pool], batch_size=16)
#         order = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)
#         return [pool[i] for i in order[:k]]
#     return fn
#
# variants = {"기준(재순위 없음)": lambda q: search_final(q)[0],
#             "R1 재순위(한국어 질문)": rerank_fn("ko"),
#             "R2 재순위(영어 번역)":   rerank_fn("en")}
# for name, fn in variants.items():
#     r = {s: evaluate(fn, items) for s, items in sets3.items()}
#     print(f"{name:20s}", r, "| 합계(66):", tuple(sum(v[i] for v in r.values()) for i in range(3)))

## 14. 최종 세트(golden_final, 60문항) — **잠가 둔 셀**

- 최종 세트는 **한 번만** 재는 시험지예요. 개발 중에 점수를 보고 설정을 고르면 안 돼요 (이미 진단 중 일부 설정을 최종 세트로 재서 오염된 적이 있어요. 그 점수로 구성을 고르거나 보고하지 않아요).
- v1 기준 점수는 `final_result_v1.json`에 저장돼 있어요: **31 / 46 / 51** (60문항, 오차 ±10%p 정도).
- 아래 셀들은 `assert`로 잠겨 있어요. 이미 결과 파일이 있으면 에러가 나요(정상).
- v2 측정은 **새로 만든 질문**으로 하거나, 오염을 인정하고 v1과 같은 질문으로 비교하되 그 사실을 같이 적어야 해요.

In [ ]:
# Ex17-a. (선택) v1 재현 검증: 최종 세트 순위가 저장된 v1과 같은지 (0이어야 해요)
# 이 검증은 7B 번역 기준이라 14B 번역에서는 달라지는 게 정상이에요. (참고용)
# saved = {x["q"]: x["rank"] for x in json.load(open(f"{DATA}/final_result_v1.json", encoding="utf-8"))["rows"]}
# diff = [g["q"] for g in tqdm(golden_final) if rank_in(search_final(g["q"])[0], g)[0] != saved[g["q"]]]
# print("순위가 달라진 문항:", len(diff), "/ 60")

In [ ]:
# Ex17-b. 개발셋과 최종 세트의 정답 겹침 확인 (점수는 재지 않음)
dev_wants = {norm(w) for s in (golden, golden_ko, golden_new) for g in s for w in wants_of(g)}
dup = [(g["q"], w) for g in golden_final for w in wants_of(g) if norm(w) in dev_wants]
print(len(golden_final), "문항 | 개발 세트와 정답이 겹침:", dup)

In [ ]:
# Ex17-c. 최종 검색 평가 (v1 결과가 이미 있어서 잠겨 있어요. 새 버전을 잴 때만 파일 이름을 바꿔 쓰세요)
final_path = f"{DATA}/final_result_v1.json"
assert not os.path.exists(final_path), "이미 최종 평가를 했어요. 다시 재려면 파일 이름(v2 등)을 바꾸고, 이유를 기록해 두세요."

rows, t0 = [], time.time()
for g in tqdm(golden_final):
    found = search_final(g["q"])[0]
    r, titles = rank_in(found, g)                 # 정답(want)은 채점에서만 사용
    rows.append({"q": g["q"], "want": g["want"], "rank": r, "top3": titles[:3]})
sec = (time.time() - t0) / len(golden_final)
c = counts([x["rank"] for x in rows])
print(f"최종 세트 (1등 / 3등 안 / 5등 안): {c} / {len(rows)} | 질문당 {sec:.1f}초")
json.dump({"config": "일반 번역 + 용어 선택지 번역, 원문+번역2 벡터, 번역 BM25, 가중 RRF (num_ctx 4096)",
           "counts": c, "sec_per_q": sec, "rows": rows},
          open(final_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
# Ex17-d. 최종 세트 60개의 실제 답변 저장 (사람이 판정할 용도, 정답 제목은 들어가지 않음)
ans_path = f"{DATA}/final_answers_v1.json"
assert not os.path.exists(ans_path), "이미 저장돼 있어요(v1 답변 판정 기준). 새로 만들 때는 다른 파일 이름을 쓰세요."
answers = []
for g in tqdm(golden_final):
    r = rag_answer(g["q"])
    answers.append({"q": g["q"], "answer": r["answer"], "sources": r["sources"], "queries": r["queries"]})
json.dump(answers, open(ans_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(len(answers), "개 저장")

## 정리 체크리스트 (다음에 할 일)

**A. 재현성**: `GEN_SEED=42`, 평가는 `temperature 0`, 챗봇 재시도만 seed를 바꿈. 같은 문항이라도 ±1문항 차이는 잡음으로 봐요. 모델을 올린 첫 호출은 워밍업. `OLLAMA_KEEP_ALIVE=1h`로 중간에 내려가는 것 방지.
**B. 평가**: 개발셋(38)은 디버깅·회귀용(오염됨), 새 세트는 만들지 않기로 함. golden_final은 v2로 **한 번만** 재고 오염 사실을 같이 적기. 지표는 자동 검사만(깨짐·혼합·분수·숫자·원문 검증), 용어 누락은 참고용.
**C. 용어표**: v1은 룰북 기준으로 한 번 만들고 고정(`term_map_v1.json`). 결과를 보고 용어를 추가하지 않기. v2 점검 목록: 기술(Skill) 18개, 크기(Tiny~Gargantuan) 6개, 덮개 3종(Half/Three-Quarters/Total Cover), 은신(Hide) 등.
**D. 파이프라인**: 선택 → 검증/거절(정책 A) → 설명 → 규칙 검사 → 충실성. 알려진 약점: Hide/Grappled류의 미묘한 의미 변화(분수 뒤집힘, 한정어 누락)는 못 잡을 수 있음.
**E. 배포**: 14B ↔ bge-m3 교체 시간(턴당 ~13초), 입력 분류기, 시나리오 RAG, FastAPI+ngrok. 챗봇 화면의 출처 표시는 팀 UX 결정.
**F. 한계**: 사용자가 원문으로 확인할 수 없으면 검사가 못 잡는 오류는 그대로 나감 → Ex16-d의 거절 비율과 함께 팀에 공유.